In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: ASSOCIATION OF RELIGION DATA ARCHIVES (ARDA) - US GROUPS
# 
# ARCHITECTURE NOTE: ARDA provides a granular taxonomy of US religious groups.
# This script uses Strategy B (Scraping) to scan the 'AllGroups' table.
# 
# UPDATED SSSOM ALIGNMENT STRATEGY: 
# 1. Root Node: Synthesizes a 'US_RG' root node to segregate this ontology.
# 2. Family Nodes: Dynamically harvests and synthesizes intermediate Family nodes 
#    (prefixed with 'F', e.g., F116) as children of the Root Node.
# 3. Group Nodes: Assigned the Family's ID as their Parent_ID.
# 4. Hierarchy_Path: US Religious Groups > [Family] > [Group Name].
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
PILOT_LIMIT = None  

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ARDA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="ARDA US Religious Groups",
    fallback_uri="https://www.thearda.com/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}_us_groups.csv")

DOMAIN = "https://www.thearda.com"
INDEX_URL = f"{DOMAIN}/us-religion/group-profiles/groups"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0'}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: {len(processed_ids)} entities already processed.")
else:
    processed_ids = set()

# Helper to write rows to CSV
def write_row(data_dict):
    clean_row = finalize_row(data_dict)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    processed_ids.add(data_dict['Concept_ID'])

# --- 4. Main Extraction Loop ---
try:
    # 4a. Initialize Root Node
    ROOT_ID = "US_RG"
    if ROOT_ID not in processed_ids:
        print("[*] Synthesizing 'US Religious Groups' root node...")
        write_row({
            'Source_System': SOURCE_NAME,
            'Primary_Label': "US Religious Groups",
            'CURIE': f"{SOURCE_PREFIX}:{ROOT_ID}",
            'Concept_Type': 'skos:Collection',
            'Hierarchy_Path': "US Religious Groups",
            'Concept_ID': ROOT_ID,
            'Status': "active"
        })

    # 4b. Fetch Group Index
    print(f"[*] Fetching Group Index from {INDEX_URL}...")
    response = requests.get(INDEX_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    table = soup.find('table', id='AllGroups')
    if not table:
        print("[!] Could not find table with id='AllGroups'.")
        sys.exit(1)
        
    tbody = table.find('tbody')
    rows = tbody.find_all('tr') if tbody else table.find_all('tr')[1:]

    count = 0
    for row in rows:
        if PILOT_LIMIT and count >= PILOT_LIMIT:
            break
            
        cols = row.find_all('td')
        if len(cols) < 3: continue
        
        # Extract Group Data
        name_link = cols[0].find('a')
        if not name_link: continue
        group_href = name_link.get('href', '')
        gid_match = re.search(r'[D|gid]=(\d+)', group_href)
        if not gid_match: continue
        gid = gid_match.group(1)
        group_name = name_link.text.strip()
        
        # Extract Family Data
        family_link = cols[2].find('a')
        if family_link:
            family_href = family_link.get('href', '')
            fid_match = re.search(r'F=(\d+)', family_href)
            fid = fid_match.group(1) if fid_match else "UNKNOWN"
            family_name = family_link.text.strip()
            family_uri = f"{DOMAIN}{family_href}" if fid_match else ""
        else:
            fid = "UNKNOWN"
            family_name = cols[2].text.strip() if cols[2].text else "Unclassified"
            family_uri = ""

        tradition = cols[1].text.strip() if cols[1].text else "Unclassified"
        family_cid = f"F{fid}" if fid != "UNKNOWN" else "F_UNKNOWN"

        # 4c. Process Family Node (If new)
        if family_cid not in processed_ids and family_cid != "F_UNKNOWN":
            print(f"  [+] Synthesizing new Family Node: {family_name} (ID:{family_cid})")
            write_row({
                'Source_System': SOURCE_NAME,
                'Primary_Label': family_name,
                'CURIE': f"{SOURCE_PREFIX}:{family_cid}",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': f"US Religious Groups > {family_name}",
                'Description': f"A structural family category within the {tradition} tradition.",
                'Parent_IDs': ROOT_ID,
                'Concept_ID': family_cid,
                'URI': family_uri,
                'Status': "active"
            })

        if gid in processed_ids: continue

        # 4d. Process Group Node
        landing_page_url = f"{DOMAIN}{group_href}"
        print(f"[{count+1}] Fetching details for Group: {group_name} (ID:{gid})...")
        
        # Nested Crawl for Description
        profile_text = ""
        try:
            p_res = requests.get(landing_page_url, headers=HEADERS, timeout=10)
            if p_res.status_code == 200:
                p_soup = BeautifulSoup(p_res.text, 'html.parser')
                body = p_soup.find('div', itemprop='articleBody')
                if body:
                    desc_tag = body.find('b', string=re.compile(r'Description:', re.I))
                    if desc_tag and desc_tag.next_sibling:
                        clean_text = str(desc_tag.next_sibling).strip()
                        clean_text = re.sub(r'\s+', ' ', clean_text)
                        profile_text = clean_text.strip('"').strip("'")
        except Exception as e:
            print(f"    [!] Failed to fetch profile page for {gid}: {e}")
        
        hierarchy_path = f"US Religious Groups > {family_name} > {group_name}"
        final_description = f"{tradition} | {family_name} | {profile_text if profile_text else ''}"
        
        write_row({
            'Source_System': SOURCE_NAME,
            'Primary_Label': group_name,
            'CURIE': f"{SOURCE_PREFIX}:{gid}",
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': hierarchy_path,
            'Description': final_description.strip(),
            'Parent_IDs': family_cid if family_cid != "F_UNKNOWN" else ROOT_ID, 
            'Concept_ID': gid,
            'URI': landing_page_url, 
            'Status': "active"
        })
        
        count += 1
        time.sleep(1.5)

except Exception as e:
    print(f"[!] Error: {e}")

print(f"\n[COMPLETE] Script finished. Processed US Group entities.")

In [ ]:
# ==============================================================================
# CELL 2 / PHASE 1: ASSOCIATION OF RELIGION DATA ARCHIVES (ARDA) - WORLD TREES
# 
# ARCHITECTURE NOTE: ARDA visualizes World Religions using the GoJS library.
# The structural taxonomy is embedded as raw JSON arrays inside the HTML <script>.
# 
# UPDATED SSSOM ALIGNMENT STRATEGY: 
# 1. Root Node: Synthesizes a 'WORLD_TREES' root node to segregate this ontology.
# 2. Hierarchy_Path: World Religions Family Tree > [Lineage...] > [Concept Name].
# 3. Parent_IDs: Top-level GoJS nodes are assigned 'WORLD_TREES' as their parent. 
#    Child nodes retain their 'W'-prefixed parent IDs (e.g., W5070).
# ==============================================================================

import os
import sys
import re
import json
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ARDA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="ARDA World Religions",
    fallback_uri="https://www.thearda.com/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_arda_world_trees.csv")

INDEX_URL = "https://www.thearda.com/world-religion/family-trees"
DOMAIN = "https://www.thearda.com"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0'}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: {len(processed_ids)} groups already processed.")
else:
    processed_ids = set()

# Helper Function: Ancestry Resolution
def get_absolute_path(node_key, nodes_dict, parents_map, depth=0):
    """Recursively builds the breadcrumb path from the root of the specific tree."""
    if depth > 15: # Prevent infinite loops in cyclic graphs
        return nodes_dict.get(node_key, {}).get("Name", "Unknown")
        
    if node_key not in parents_map or not parents_map[node_key]:
        return nodes_dict.get(node_key, {}).get("Name", "Unknown")
        
    # Polyhierarchy handling: default to the first parent for the visual path
    primary_parent = parents_map[node_key][0]
    parent_path = get_absolute_path(primary_parent, nodes_dict, parents_map, depth + 1)
    current_name = nodes_dict.get(node_key, {}).get("Name", "Unknown")
    
    return f"{parent_path} > {current_name}"

# --- 4. Main Extraction Loop ---
try:
    all_rows = []
    
    # 4a. Initialize Root Node
    ROOT_ID = "WORLD_TREES"
    ROOT_LABEL = "World Religions Family Tree"
    
    if ROOT_ID not in processed_ids:
        print(f"[*] Synthesizing '{ROOT_LABEL}' root node...")
        root_data = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': ROOT_LABEL,
            'CURIE': f"{SOURCE_PREFIX}:{ROOT_ID}",
            'Concept_Type': 'skos:Collection',
            'Hierarchy_Path': ROOT_LABEL,
            'Concept_ID': ROOT_ID,
            'Status': "active"
        }
        all_rows.append(finalize_row(root_data))
        processed_ids.add(ROOT_ID)

    # 4b. Fetch Tree URLs
    print(f"[*] Fetching World Religion Tree Index from {INDEX_URL}...")
    response = requests.Session().get(INDEX_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find links to the individual family trees and filter out abbreviated versions
    tree_links = soup.find_all('a', href=re.compile(r'family-trees\?W=1&F=\d+'))
    tree_urls = list(set([DOMAIN + a['href'] for a in tree_links if 'showAbbrev' not in a['href']]))
    print(f"[*] Found {len(tree_urls)} World Religion Trees to process.")

    # 4c. Process Each Tree
    for url in tree_urls:
        print(f"\n[*] Processing Tree: {url}")
        
        # 3-try retry loop with backoff for API politeness
        for attempt in range(3):
            try:
                tree_res = requests.get(url, headers=HEADERS, timeout=15)
                tree_res.raise_for_status()
                break
            except requests.exceptions.RequestException as e:
                print(f"    [!] Attempt {attempt + 1} failed: {e}")
                time.sleep(2 ** attempt)
        else:
            print(f"    [!] Skipping tree after 3 failed attempts: {url}")
            continue
        
        # Regex to target the JSON arrays inside the createDiagram() JavaScript function
        js_match = re.search(r'createDiagram\(.*?,\s*".*?",\s*(\[.*?\]),\s*(\[.*?\])', tree_res.text, re.DOTALL)
        
        if not js_match:
            print("    [!] Could not locate GoJS data array in HTML source. Skipping.")
            continue
            
        try:
            nodes_data = json.loads(js_match.group(1))
            edges_data = json.loads(js_match.group(2))
        except json.JSONDecodeError as e:
            print(f"    [!] JSON Parse Error: {e}")
            continue
            
        # Build relational maps
        nodes_dict = {n['Key']: n for n in nodes_data}
        parents_map = {}
        
        for edge in edges_data:
            child = edge['ChildKey']
            parent = edge['ParentKey']
            if child not in parents_map:
                parents_map[child] = []
            parents_map[child].append(parent)
            
        # Transform Nodes into standard schema
        for key, node in nodes_dict.items():
            cid = f"W{key}"
            if cid in processed_ids: continue
                
            name = str(node.get('Name', '')).strip()
            
            # --- Clean Description ---
            raw_desc = str(node.get('Desc', ''))
            clean_desc = re.sub(r'^.*?\)\s*:\s*', '', raw_desc).strip()
            clean_desc = re.sub(r'\s+', ' ', clean_desc)
            clean_desc = clean_desc.strip('"').strip("'").strip()
            
            # --- Parent Resolution ---
            p_keys = parents_map.get(key, [])
            if not p_keys:
                # Top-level nodes default to the master root node
                parent_ids_str = ROOT_ID
            else:
                parent_ids_str = " | ".join([f"W{p}" for p in p_keys])
            
            # --- Hierarchy Path Construction ---
            raw_path = get_absolute_path(key, nodes_dict, parents_map)
            hierarchy_path = f"{ROOT_LABEL} > {raw_path}"
            
            extracted_data = {
                'Source_System': SOURCE_NAME,
                'Primary_Label': name,
                'CURIE': f"{SOURCE_PREFIX}:{cid}",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': hierarchy_path,
                'Description': clean_desc,
                'Parent_IDs': parent_ids_str, 
                'Concept_ID': cid,
                'URI': "", 
                'Status': "active"
            }
            
            clean_row = finalize_row(extracted_data)
            all_rows.append(clean_row)
            processed_ids.add(cid)
            
        print(f"    [+] Extracted {len(nodes_data)} concepts.")
        time.sleep(1) # Be polite between trees

    # --- 5. Export to Bronze Layer ---
    if all_rows:
        export_df = pd.DataFrame(all_rows)[COLUMN_ORDER]
        export_df.to_csv(
            raw_ingest_file, mode='a', index=False, 
            header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        print(f"\n[COMPLETE] Successfully wrote {len(all_rows)} World Religion concepts to {os.path.basename(raw_ingest_file)}.")
    else:
        print("\n[*] No new records to write.")

except Exception as e:
    print(f"[!] Critical Error: {e}")

In [ ]:
# ==============================================================================
# CELL 3 / PHASE 1: ASSOCIATION OF RELIGION DATA ARCHIVES (ARDA) - MEASUREMENTS
# 
# ARCHITECTURE NOTE: ARDA organizes sociological survey concepts into more than 
# 160 "Single-Item Measures". This script scrapes the index table and the category 
# dropdown list to harvest the Concept, Category, and native IDs.
# 
# UPDATED SSSOM ALIGNMENT STRATEGY: 
# 1. Root Node: Synthesizes an 'M_ROOT' node ("Measurements").
# 2. Category Nodes: Dynamically parses the <select> tag to find native category IDs. 
#    Prefixes them with 'MC' (e.g., MC1) and sets their parent to M_ROOT.
# 3. Concept Nodes: Assigned the 'MC' Category ID as their Parent_ID.
# 4. Hierarchy_Path: Measurements > [Category] > [Concept Name].
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
PILOT_LIMIT = None  

notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in the config directory.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "ARDA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Association of Religion Data Archives",
    fallback_uri="https://www.thearda.com/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}_measurements.csv")

DOMAIN = "https://www.thearda.com"
INDEX_URL = f"{DOMAIN}/data-archive/measurements"
HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0'}

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: {len(processed_ids)} measurements already processed.")
else:
    processed_ids = set()

# Helper to write rows to CSV
def write_row(data_dict):
    clean_row = finalize_row(data_dict)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    processed_ids.add(data_dict['Concept_ID'])

# --- 4. Main Extraction Loop ---
try:
    # 4a. Initialize Root Node
    ROOT_ID = "M_ROOT"
    ROOT_LABEL = "Measurements"
    if ROOT_ID not in processed_ids:
        print(f"[*] Synthesizing '{ROOT_LABEL}' root node...")
        write_row({
            'Source_System': SOURCE_NAME,
            'Primary_Label': ROOT_LABEL,
            'CURIE': f"{SOURCE_PREFIX}:{ROOT_ID}",
            'Concept_Type': 'skos:Collection',
            'Hierarchy_Path': ROOT_LABEL,
            'Concept_ID': ROOT_ID,
            'Status': "active"
        })

    # 4b. Fetch Measurements Index
    print(f"[*] Fetching Measurements Index from {INDEX_URL}...")
    response = requests.get(INDEX_URL, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 4c. Extract Category Dictionary
    category_map = {}
    select_tag = soup.find('select', id='category_concept')
    if select_tag:
        for option in select_tag.find_all('option'):
            val = option.get('value')
            text = option.text.strip()
            if val and val != "0":
                category_map[text] = val
        print(f"[*] Successfully mapped {len(category_map)} native Measurement Categories.")
    else:
        print("[!] Warning: Could not find category_concept dropdown. Categories will lack native IDs.")

    # 4d. Locate Main Table
    table = soup.find('table', id='SingleItemMeasures')
    if not table:
        print("[!] Could not find table with id='SingleItemMeasures'.")
        sys.exit(1)
        
    tbody = table.find('tbody')
    rows = tbody.find_all('tr') if tbody else table.find_all('tr')[1:]

    count = 0
    for row in rows:
        if PILOT_LIMIT and count >= PILOT_LIMIT:
            break
            
        cols = row.find_all('td')
        if len(cols) < 2: continue
        
        # Extract Measurement Data
        name_link = cols[0].find('a')
        if not name_link: continue
        relative_href = name_link.get('href', '')
        scid_match = re.search(r'scid=(\d+)', relative_href)
        if not scid_match: continue
        
        raw_id = scid_match.group(1)
        cid = f"M{raw_id}" 
        concept_name = name_link.text.strip()
        
        # Extract Category Data
        category_name = cols[1].text.strip() if cols[1].text else "Uncategorized"
        ccid_raw = category_map.get(category_name)
        category_cid = f"MC{ccid_raw}" if ccid_raw else "MC_UNKNOWN"

        # 4e. Process Category Node (If new)
        if category_cid not in processed_ids and category_cid != "MC_UNKNOWN":
            print(f"  [+] Synthesizing new Category Node: {category_name} (ID:{category_cid})")
            write_row({
                'Source_System': SOURCE_NAME,
                'Primary_Label': category_name,
                'CURIE': f"{SOURCE_PREFIX}:{category_cid}",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': f"{ROOT_LABEL} > {category_name}",
                'Parent_IDs': ROOT_ID,
                'Concept_ID': category_cid,
                'Status': "active"
            })

        if cid in processed_ids: continue

        # 4f. Process Measurement Node
        landing_page_url = f"{DOMAIN}{relative_href}"
        print(f"[{count+1}] Fetching details for Measurement: {concept_name} (ID:{cid})...")
        
        # Nested Crawl for Description
        profile_text = ""
        try:
            p_res = requests.get(landing_page_url, headers=HEADERS, timeout=10)
            if p_res.status_code == 200:
                p_soup = BeautifulSoup(p_res.text, 'html.parser')
                body = p_soup.find('div', itemprop='articleBody')
                
                if body:
                    h1_tag = body.find('h1')
                    if h1_tag:
                        desc_parts = []
                        for sibling in h1_tag.next_siblings:
                            if getattr(sibling, 'name', None) in ['i', 'hr', 'div', 'table']:
                                break
                            if isinstance(sibling, str):
                                desc_parts.append(sibling)
                                
                        raw_text = " ".join(desc_parts).strip()
                        clean_text = re.sub(r'\s+', ' ', raw_text)
                        profile_text = clean_text.strip('"').strip("'")
                        
        except Exception as e:
            print(f"    [!] Failed to fetch profile page for {cid}: {e}")
        
        hierarchy_path = f"{ROOT_LABEL} > {category_name} > {concept_name}"
        
        write_row({
            'Source_System': SOURCE_NAME,
            'Primary_Label': concept_name,
            'CURIE': f"{SOURCE_PREFIX}:{cid}",
            'Concept_Type': 'skos:Concept',
            'Hierarchy_Path': hierarchy_path,
            'Description': profile_text,
            'Parent_IDs': category_cid if category_cid != "MC_UNKNOWN" else ROOT_ID, 
            'Concept_ID': cid,
            'URI': landing_page_url, 
            'Status': "active"
        })
        
        count += 1
        time.sleep(1.5)

except Exception as e:
    print(f"[!] Error: {e}")

print(f"\n[COMPLETE] Script finished. Processed ARDA Measurement entities.")